# Lab 3: Do Size and Value Explain Returns That Beta Cannot?

*Financial Engineering in Practice*, Chapter 3.

**What you will explore:** how much additional factors add to the CAPM in explaining differences in portfolio returns, and how testing many candidate factors can produce false discoveries.

You will estimate the time-series regressions of Fama and French (1993), run a multiple-testing experiment in the spirit of Harvey, Liu and Zhu (2016), and ask whether the factor premiums are large enough to matter. Before each step, write down a prediction.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab03

SOURCE = "french"   # "french" downloads real data; "sim" uses fake data to test the code
OUT = Path("report")
OUT.mkdir(exist_ok=True)
data = lab03.load(SOURCE)
SIM = data["simulated"]
Y, F = lab03.panel(data, start="1963-07")
print(Y.index[0], "to", Y.index[-1], "| simulated:", SIM)

## Part A: the CAPM against the three-factor model

**Predict first:** which portfolio will have the largest CAPM alpha? Will adding SMB and HML shrink the alphas toward zero, or only raise the R²?

In [ ]:
a = lab03.part_a(Y, F)
lab03.fig_alpha_grid(a, OUT, SIM); display(Image(OUT / "fig3_1_alpha_grid.png"))
for m in ("CAPM", "Three-factor"):
    print(m, a[m]["summary"], "GRS p =", round(a[m]["grs"]["p_value"], 4))

In [ ]:
lab03.fig_r2(a, OUT, SIM); lab03.fig_predicted(a, OUT, SIM)
display(Image(OUT / "fig3_2_r2.png")); display(Image(OUT / "fig3_3_predicted.png"))
a["Three-factor"]["table"].round(3)

A higher R² says the model tracks each portfolio's month-to-month movements. The alphas, and the GRS test that all 25 are zero, ask whether it explains the *average* returns.

## Part B: 1,000 useless factors

**Predict first:** of 1,000 factors whose true premium is zero, how many will have |t| > 1.96? How many |t| > 3?

In [ ]:
b = lab03.part_b(n_months=len(F), n_factors=1000, n_reps=200)
lab03.fig_false_discoveries(b, OUT, SIM); display(Image(OUT / "fig3_4_false_discoveries.png"))
b["table"].round(1)

With correlated factors the average count is about the same, but some runs produce a flood of "discoveries" at once. That is closer to a field in which many researchers test related ideas on the same data.

## Part C: statistical and economic significance

**Predict first:** which factor has the highest Sharpe ratio? Is the size premium as strong after 1990 as before?

In [ ]:
c = lab03.part_c(F, turnover=0.03)
lab03.fig_premiums(c, OUT, SIM); display(Image(OUT / "fig3_5_premiums.png"))
c["table"].round(3)

In [ ]:
# The one-way trading cost that would erase each long-short premium, at an assumed 3% monthly turnover per leg
c["break_even"][["factor", "premium_annual", "assumed_turnover", "break_even_one_way_cost"]].round(4)

## Questions to think about

Using your own numbers, write two or three sentences:

1. Do SMB and HML explain the 25 portfolios better than the market alone, judged by the alphas and the GRS test?
2. What hurdle would you require of a new factor, given Part B?
3. Are the premiums large and stable enough to matter after costs?